In [17]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
from unidecode import unidecode
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pds = pandas series  
df = DataFrame

# Preparando o Data Frame

In [18]:
dados = pd.read_excel('WDI_EXCEL/WDIEXCEL.xlsx')

In [83]:
dados.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.488497,18.001597,18.558234,19.043572,19.586457,20.192064,20.828814,21.372164,22.100884,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,6.811504,7.096003,7.406706,7.666648,8.020952,8.403358,8.718306,9.097176,9.473374,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,38.152090,38.488233,38.779953,39.068462,39.445526,39.818645,40.276374,40.687817,41.211606,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,31.871956,33.922276,38.859598,40.223744,43.035073,44.390861,46.282371,48.127211,48.742043,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.672943,16.527554,24.627753,25.432092,27.061929,29.154282,31.022083,32.809138,33.760782,NaN


In [84]:
dados.columns = [col.strip().replace("_", " ") for col in dados.columns]

In [85]:
print(dados.columns)


Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023'],
      dtype='object')


In [86]:
dados_long = dados.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="Value"
)

In [87]:
dados_long["Year"] = dados_long["Year"].astype(int)

In [88]:
dados_long.dtypes

Country Name       object
Country Code       object
Indicator Name     object
Indicator Code     object
Year                int64
Value             float64
dtype: object

## Separando por paises e comaçando a olhar o data frame do Brasil

Separação feita desta maneira para uma posssivel extrapolação para demais paises futuramente

In [89]:
paises_unicos = dados_long["Country Name"].unique()

sub_dfs_por_pais = {
    pais: grupo.drop(columns=["Country Code", "Country Name"]).reset_index(drop=True)
    for pais, grupo in dados_long.groupby("Country Name")
}

brasil = sub_dfs_por_pais["Brazil"]
print("\nDados do Brasil:")
brasil.head()


Dados do Brasil:


,Indicator Name,Indicator Code,Year,Value
0,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1960,NaN
1,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1960,NaN
2,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,1960,NaN
3,Access to electricity (% of population),EG.ELC.ACCS.ZS,1960,NaN
4,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,1960,NaN


# Analise do Brasil

## Explorando e preparando o data frame para analise

In [206]:
# 1. Filtrar apenas as colunas importantes para a pivotagem
df_brasil = brasil[["Year", "Indicator Name", "Value"]].copy()

# 2. Pivotar para ficar no formato "wide" (uma coluna por indicador)
df_brasil_wide = df_brasil.pivot(
    index="Year",
    columns="Indicator Name",
    values="Value"
)

# 3. Ordenar pelas datas (anos) e exibir as primeiras linhas
df_brasil_wide.sort_index(inplace=True)
df_brasil_wide.head()

Indicator Name,ARI treatment (% of children under 5 taken to a health provider),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)","Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)",...,Women who believe a husband is justified in beating his wife (any of five reasons) (%),Women who believe a husband is justified in beating his wife when she argues with him (%),Women who believe a husband is justified in beating his wife when she burns the food (%),Women who believe a husband is justified in beating his wife when she goes out without telling him (%),Women who believe a husband is justified in beating his wife when she neglects the children (%),Women who believe a husband is justified in beating his wife when she refuses sex with him (%),Women who were first married by age 15 (% of women ages 20-24),Women who were first married by age 18 (% of women ages 20-24),Women's share of population ages 15+ living with HIV (%),Young people (ages 15-24) newly infected with HIV
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [232]:
pds_br_tps = df_brasil_wide.dtypes
for i ,formato in pds_br_tps.items():
    if formato != 'float64' and formato != 'int64':
        print(i, formato)

Verificando quantidade de nulos por indicador 

In [208]:
# Contar valores nulos por linha
pds_nulos_por_indicador = df_brasil_wide.isna().sum(axis=0)

# Exibir o resultado
print("Quantidade de valores nulos por linha (ano):")
print(pds_nulos_por_indicador.sort_values(ascending=False))

Quantidade de valores nulos por linha (ano):
Indicator Name
Women who believe a husband is justified in beating his wife when she refuses sex with him (%)       64
Vitamin A supplementation coverage rate (% of children ages 6-59 months)                             64
Young people (ages 15-24) newly infected with HIV                                                    64
Educational attainment, at least completed post-secondary, population 25+, male (%) (cumulative)     64
Educational attainment, at least completed post-secondary, population 25+, total (%) (cumulative)    64
                                                                                                     ..
Terms of trade adjustment (constant LCU)                                                              0
Agriculture, forestry, and fishing, value added (constant 2015 US$)                                   0
Agriculture, forestry, and fishing, value added (constant LCU)                                        0
Agri

In [ ]:
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadores_remover = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor >= 24: #Garantir ao menos 40 anos de dados
        indicadores_remover.append(indicador)
print(len(indicadores_remover))

Indicadores a serem removidos:
954


In [210]:
df_corr = df_brasil_wide[indicadores_remover + ["Population growth (annual %)"]]

# Remove linhas com nulos para correlação
df_corr = df_corr.dropna()

# Calcula correlação de Pearson
correlacoes = df_corr.corr()

# Filtra a correlação apenas com 'Urban population'
correlacoes_com_target = correlacoes["Population growth (annual %)"].drop("Population growth (annual %)").sort_values(key=abs, ascending=False)

print(correlacoes_com_target.sort_values(ascending=False))

Indicator Name
Women who believe a husband is justified in beating his wife when she refuses sex with him (%)      NaN
Vitamin A supplementation coverage rate (% of children ages 6-59 months)                            NaN
Young people (ages 15-24) newly infected with HIV                                                   NaN
Educational attainment, at least completed post-secondary, population 25+, male (%) (cumulative)    NaN
Educational attainment, at least completed post-secondary, population 25+, total (%) (cumulative)   NaN
                                                                                                     ..
Tertiary education, academic staff (% female)                                                       NaN
Net official flows from UN agencies, IFAD (current US$)                                             NaN
Investment in energy with private participation (current US$)                                       NaN
Net official flows from UN agencies, UNTA (curren

In [211]:
df_corr = df_brasil_wide[indicadores_remover + ["Population, total"]]

# Remove linhas com nulos para correlação
df_corr = df_corr.dropna()

# Calcula correlação de Pearson
correlacoes = df_corr.corr()

# Filtra a correlação apenas com 'Urban population'
correlacoes_com_target = correlacoes["Population, total"].drop("Population, total").sort_values(key=abs, ascending=False)

print(correlacoes_com_target.sort_values(ascending=False))

Indicator Name
Women who believe a husband is justified in beating his wife when she refuses sex with him (%)      NaN
Vitamin A supplementation coverage rate (% of children ages 6-59 months)                            NaN
Young people (ages 15-24) newly infected with HIV                                                   NaN
Educational attainment, at least completed post-secondary, population 25+, male (%) (cumulative)    NaN
Educational attainment, at least completed post-secondary, population 25+, total (%) (cumulative)   NaN
                                                                                                     ..
Tertiary education, academic staff (% female)                                                       NaN
Net official flows from UN agencies, IFAD (current US$)                                             NaN
Investment in energy with private participation (current US$)                                       NaN
Net official flows from UN agencies, UNTA (curren

In [212]:
new_df_brasil_wide = df_brasil_wide.drop(columns=indicadores_remover)
new_df_brasil_wide.head()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,"Travel services (% of service imports, BoP)","Unemployment, female (% of female labor force) (national estimate)","Unemployment, male (% of male labor force) (national estimate)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,33399157.0,46.139,NaN,NaN,1.768224e-13,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35155579.0,47.122,5.125267,NaN,2.446007e-13,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,36971452.0,48.099,5.036272,NaN,3.748121e-13,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,38852223.0,49.078,4.961925,NaN,6.508432e-13,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,40792376.0,50.059,4.872991,NaN,1.247164e-12,NaN


Verificando quantidade de nulos por ano

In [213]:
# Contar valores nulos por linha
pds_nulos_por_ano = new_df_brasil_wide.isna().sum(axis=1)

# Exibir o resultado
print("Quantidade de valores nulos por linha (ano):")
print(pds_nulos_por_ano.sort_values(ascending=False))

Quantidade de valores nulos por linha (ano):
Year
1960    300
1961    271
1962    257
1963    255
1964    254
       ... 
2014      0
2008      0
2013      0
2009      0
2011      0
Length: 64, dtype: int64


In [ ]:
pds_nulos_por_ano = pds_nulos_por_ano.sort_values(ascending=False)
anos_remover = []
for indicador, valor in pds_nulos_por_ano.items():
    if valor >= (542*0.3):
        anos_remover.append(indicador)
print(len(anos_remover))

Indicadores a serem removidos:
10


In [215]:
anos_remover.sort()
anos_remover

[1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969]

In [216]:
new_df_brasil_wide = new_df_brasil_wide.drop(index=anos_remover)
new_df_brasil_wide.head()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,"Travel services (% of service imports, BoP)","Unemployment, female (% of female labor force) (national estimate)","Unemployment, male (% of male labor force) (national estimate)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1970,NaN,3.291109e+11,3.782767e+10,NaN,3450.680697,396.617704,9.121553,3.806293e+09,3.599282,1.501929e+09,...,NaN,NaN,NaN,NaN,53323573.0,55.909,4.221541,0.0,5.881536e-12,31.250
1971,11.844305,3.680918e+11,4.418975e+10,9.139582,3766.058479,452.118667,9.018367,4.390516e+09,3.599282,1.752280e+09,...,NaN,NaN,NaN,NaN,55607782.0,56.894,4.194465,0.0,7.057843e-12,31.250
1972,12.509426,4.141380e+11,5.269725e+10,9.804705,4135.309393,526.200059,9.028985,5.243614e+09,3.599282,2.090295e+09,...,NaN,NaN,NaN,3.06,57963965.0,57.879,4.149837,0.0,8.369877e-12,31.250
1973,14.291462,4.733244e+11,7.074687e+10,11.557651,4613.254011,689.534076,9.501022,7.446978e+09,3.000000,2.351424e+09,...,NaN,NaN,NaN,2.78,60385804.0,58.855,4.093252,0.0,9.772397e-12,31.250
1974,6.152843,5.024473e+11,9.352216e+10,3.638966,4781.128777,889.927177,9.749445,1.017117e+10,3.599282,3.754972e+09,...,NaN,NaN,NaN,NaN,62870949.0,59.826,4.033015,0.0,1.262268e-11,28.125


In [217]:
len(new_df_brasil_wide)

54

In [218]:
df_corr = new_df_brasil_wide

# Remove linhas com nulos para correlação
df_corr = df_corr.dropna()

# Calcula correlação de Pearson
correlacoes = df_corr.corr()

# Filtra a correlação apenas com 'Urban population'
correlacoes_com_target_total = correlacoes["Population, total"].drop("Population, total").sort_values(key=abs, ascending=False)

print(correlacoes_com_target_total.sort_values(ascending=False))

Indicator Name
Population density (people per sq. km of land area)                      1.000000
Population ages 15-64, male                                              0.999985
Population, female                                                       0.999970
Population, male                                                         0.999965
Population ages 15-64, total                                             0.999936
                                                                           ...   
Land area (sq. km)                                                            NaN
Lower secondary school starting age (years)                                   NaN
Renewable internal freshwater resources, total (billion cubic meters)         NaN
Secondary education, duration (years)                                         NaN
Surface area (sq. km)                                                         NaN
Name: Population, total, Length: 541, dtype: float64


In [219]:
indicadores_remover_total = [
    i for i in correlacoes_com_target_total.index if abs(correlacoes_com_target_total[i]) < 0.6
]
print(len(indicadores_remover_total))

141


In [220]:
# Filtra a correlação apenas com 'Urban population'
correlacoes_com_target_growth = correlacoes["Population growth (annual %)"].drop("Population growth (annual %)").sort_values(key=abs, ascending=False)

print(correlacoes_com_target_growth.sort_values(ascending=False))

Indicator Name
Population ages 00-04, male (% of male population)                       0.997626
Population ages 00-04, female (% of female population)                   0.997614
Mortality rate, neonatal (per 1,000 live births)                         0.994029
Birth rate, crude (per 1,000 people)                                     0.993607
Population ages 0-14, female (% of female population)                    0.993263
                                                                           ...   
Land area (sq. km)                                                            NaN
Lower secondary school starting age (years)                                   NaN
Renewable internal freshwater resources, total (billion cubic meters)         NaN
Secondary education, duration (years)                                         NaN
Surface area (sq. km)                                                         NaN
Name: Population growth (annual %), Length: 541, dtype: float64


In [221]:
indicadores_remover_growth = [
    i for i in correlacoes_com_target_growth.index if abs(correlacoes_com_target_growth[i]) < 0.6
]
print(len(indicadores_remover_growth))

139


In [222]:
indicadores_remover_final = list(set(indicadores_remover_total) & set(indicadores_remover_growth))
print(len(indicadores_remover_final))

136


In [223]:
final_df_brasil = new_df_brasil_wide.drop(columns=indicadores_remover_final)
final_df_brasil.head()

Indicator Name,Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),Adjusted savings: energy depletion (% of GNI),Adjusted savings: energy depletion (current US$),...,"Transport services (% of service imports, BoP)",Travel services (% of commercial service exports),Travel services (% of commercial service imports),"Travel services (% of service exports, BoP)","Travel services (% of service imports, BoP)",Urban population,Urban population (% of total population),Urban population growth (annual %),Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1970,3.291109e+11,3.782767e+10,3450.680697,396.617704,9.121553,3.806293e+09,3.599282,1.501929e+09,0.047728,1.991631e+07,...,NaN,NaN,NaN,NaN,NaN,53323573.0,55.909,4.221541,5.881536e-12,31.250
1971,3.680918e+11,4.418975e+10,3766.058479,452.118667,9.018367,4.390516e+09,3.599282,1.752280e+09,0.070111,3.413296e+07,...,NaN,NaN,NaN,NaN,NaN,55607782.0,56.894,4.194465,7.057843e-12,31.250
1972,4.141380e+11,5.269725e+10,4135.309393,526.200059,9.028985,5.243614e+09,3.599282,2.090295e+09,0.066085,3.837894e+07,...,NaN,NaN,NaN,NaN,NaN,57963965.0,57.879,4.149837,8.369877e-12,31.250
1973,4.733244e+11,7.074687e+10,4613.254011,689.534076,9.501022,7.446978e+09,3.000000,2.351424e+09,0.088562,6.941582e+07,...,NaN,NaN,NaN,NaN,NaN,60385804.0,58.855,4.093252,9.772397e-12,31.250
1974,5.024473e+11,9.352216e+10,4781.128777,889.927177,9.749445,1.017117e+10,3.599282,3.754972e+09,0.388249,4.050435e+08,...,NaN,NaN,NaN,NaN,NaN,62870949.0,59.826,4.033015,1.262268e-11,28.125


In [224]:
final_df_brasil.describe()

Indicator Name,Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),Adjusted savings: energy depletion (% of GNI),Adjusted savings: energy depletion (current US$),...,"Transport services (% of service imports, BoP)",Travel services (% of commercial service exports),Travel services (% of commercial service imports),"Travel services (% of service exports, BoP)","Travel services (% of service imports, BoP)",Urban population,Urban population (% of total population),Urban population growth (annual %),Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
count,5.200000e+01,5.200000e+01,52.000000,52.000000,52.000000,5.200000e+01,52.000000,5.200000e+01,52.000000,5.200000e+01,...,49.000000,49.000000,49.000000,49.000000,49.000000,5.400000e+01,54.000000,54.000000,5.200000e+01,54.000000
mean,9.162268e+11,6.681062e+11,5590.578090,3682.081997,14.456639,1.461168e+11,4.417439,4.287812e+10,0.823144,9.483043e+09,...,35.557483,17.097469,19.294171,16.320850,18.161931,1.256833e+08,75.951296,2.385401,5.155756e+01,57.060185
std,3.304046e+11,5.915988e+11,916.290481,2801.135806,3.945016,1.645337e+11,0.982101,4.571928e+10,0.564110,1.188440e+10,...,12.940349,9.433232,7.152482,9.000527,7.024541,4.176731e+07,9.722235,1.177507,6.612087e+01,22.270061
min,3.291109e+11,3.782767e+10,3450.680697,396.617704,9.018367,3.806293e+09,2.800000,1.501929e+09,0.047728,1.991631e+07,...,15.218423,2.366505,6.648368,2.262181,5.898268,5.332357e+07,55.909000,0.632380,5.881536e-12,28.125000
25%,6.445987e+11,1.925976e+11,4987.951719,1490.704292,11.792705,2.432008e+10,3.599282,7.718184e+09,0.324402,1.192304e+09,...,23.040131,7.535885,14.728987,7.253886,13.296869,8.921814e+07,68.359000,1.201713,1.327366e-09,33.125000
50%,9.307267e+11,4.434869e+11,5542.382954,2537.679504,13.756643,7.698704e+10,4.264303,2.063184e+10,0.723090,2.596915e+09,...,32.641346,17.807276,19.942503,17.324395,18.475186,1.301583e+08,78.675000,2.422724,2.191907e+01,50.625000
75%,1.271165e+12,1.160585e+12,6433.397369,5848.704142,15.720799,2.527057e+11,5.217500,8.456528e+10,1.305327,1.550654e+10,...,45.488234,23.862175,24.342780,22.045219,22.862175,1.628768e+08,84.262250,3.513118,9.463679e+01,81.875000
max,1.435186e+12,2.003637e+12,7230.947942,10260.080844,23.062757,5.187513e+11,6.281311,1.399386e+11,2.263552,3.886728e+10,...,56.829073,39.869707,35.950176,39.080460,34.497130,1.853562e+08,87.788000,4.221541,2.752110e+02,85.000000


In [225]:
pds_nulos_por_indicador = final_df_brasil.isna().sum(axis=0)
print(pds_nulos_por_indicador.sort_values(ascending=False))

Indicator Name
Labor force participation rate for ages 15-24, total (%) (national estimate)                  14
Ratio of female to male labor force participation rate (%) (national estimate)                14
Labor force participation rate, total (% of total population ages 15+) (national estimate)    14
Net bilateral aid flows from DAC donors, Denmark (current US$)                                14
Labor force participation rate, male (% of male population ages 15+) (national estimate)      14
                                                                                              ..
GDP (current LCU)                                                                              0
GDP (current US$)                                                                              0
GDP deflator (base year varies by country)                                                     0
GDP per capita (constant 2015 US$)                                                             0
General governm

In [226]:
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadores_analisar = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor >= 10 :
        indicadores_analisar.append(indicador)
    elif valor <= 5:
        final_df_brasil[indicador] = final_df_brasil[indicador].interpolate(method='linear', limit_direction='both') 
print("Indicadores a serem analisados:")
print(len(indicadores_analisar))

Indicadores a serem analisados:
31
